# Same AOI across providers: MPC + CDSE

Demonstrates the endpoint/signer abstraction: the **same** request shape pulls from Microsoft Planetary Computer (Azure SAS URL signing) and Copernicus Data Space (S3 credentials) over one AOI. Only `data_source` (and CDSE keys) change. The MPC pull runs live (degrades gracefully offline); the CDSE pull is a recipe (needs S3 keys).

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
print("MPC signer:", cat.get_endpoint("planetary-computer").signer)
print("CDSE signer:", cat.get_endpoint("cdse").signer)
print("MPC S2 id:", cat.resolve("planetary-computer", "sentinel-2-l2a"))

## Planetary Computer — Sentinel-2 (SAS signing, no account, live)

MPC serves Sentinel-2 L2A bands as `B04`/`B03`/`B02`; assets are signed with an Azure SAS token via the `planetary-computer` SDK.

In [ ]:
import tempfile
from earthlens.earthlens import EarthLens

aoi = dict(lat_lim=[40.40, 40.45], lon_lim=[-3.72, -3.67], start="2024-06-01", end="2024-06-20")
try:
    mpc = EarthLens(
        data_source="planetary-computer",
        variables={"sentinel-2-l2a": ["B04", "B03", "B02"]},
        path=tempfile.mkdtemp(), max_items=1, **aoi,
    ).download()
except Exception as exc:  # noqa: BLE001 - stays runnable offline
    mpc = []
    print("download skipped (needs network access):", exc)
[p.name for p in mpc]

## CDSE — Sentinel-2 (S3 keys; recipe)

Set `CDSE_S3_ACCESS_KEY` / `CDSE_S3_SECRET_KEY` first (see Authentication), then the identical shape pulls from CDSE — the backend picks `CdseS3Signer` from the catalog instead of pyramids' `PlanetaryComputerSigner`:

```python
cdse = EarthLens(
    data_source="cdse",
    variables={"sentinel-2-l2a": ["B04", "B03", "B02"]},
    path="out/cdse", max_items=1, **aoi,
).download()
```

Both write COGs over the identical AOI; reproject/stack them with pyramids to compare providers over the same scene.